# Seccion 2: Preprocesado de datos

¿Escoger solo frontales o todas las radiografias?  
¿Eliminar patologias con menos casos?  
¿Que hacemos con los pacientes que tienen un ingente número de radiográfias?


# 3. Estrategias de Preprocesamiento en Deep Learning Médico

A diferencia del Machine Learning clásico para datos tabulares (donde se aplican técnicas como `SMOTE` o `StandardScaler`), el preprocesamiento en **Visión Artificial Médica** se divide en dos grandes ejes: el tratamiento de los **metadatos** (el *DataFrame*) y el procesamiento espacial de los **tensores** (las imágenes).

A continuación, se detalla el catálogo de técnicas aplicables a la cohorte CheXpert:

### 3.1. Transformación de Metadatos y Etiquetas (Dataframe)
Antes de cargar las imágenes, es necesario traducir el lenguaje médico y la incertidumbre a un formato matemático estricto que la red neuronal pueda computar en su función de pérdida.

* **Estrategias de Imputación de Incertidumbre (-1.0):**
    * **U-Zeroes:** Convertir `-1.0` y `NaN` a `0.0`. (Asunción clínica: "si hay duda o no se menciona explícitamente, se asume sano").
    * **U-Ones:** Convertir `-1.0` a `1.0`. (Asunción clínica conservadora: "ante la duda, se asume patología para evitar falsos negativos").
    * **Label Smoothing:** Sustituir el `-1.0` por un valor probabilístico intermedio (ej. `0.6`), indicando a la red un 60% de probabilidad de patología.

### 3.2. Partición Estructural y Segura (Data Splitting)
Técnicas orientadas a evitar el **Data Leakage** (Fuga de Datos) detectado en el EDA, garantizando una evaluación honesta del modelo.

* **Group Shuffle Split (Aislamiento por Paciente):** Agrupar todas las radiografías utilizando el identificador único del paciente (`Patient`). Esto garantiza que todas las visitas médicas de un paciente crónico vayan íntegramente al conjunto de *Train* o al de *Validation*, impidiendo que la red memorice su anatomía.
* **Estratificación Demográfica:** Asegurar que la proporción de hombres y mujeres se mantenga constante entre los conjuntos de entrenamiento y validación para poder auditar el *Fairness* (equidad) del algoritmo.

### 3.3. Preprocesamiento Base de la Imagen (Transformaciones Deterministas)
Estas transformaciones se aplican **siempre**, tanto en entrenamiento como en validación/test, para estandarizar la entrada geométrica y de color a la red convolucional.

* **Redimensionamiento Espacial (Resize):** Escalar matrices de dimensiones variables originales a un tensor estricto requerido por arquitecturas como DenseNet o ResNet (típicamente `224x224` o `320x320` píxeles).
* **Escalado de Tensores:** Convertir los valores de los píxeles del rango entero `[0, 255]` a coma flotante `[0.0, 1.0]`.
* **Normalización Z-Score (ImageNet Standardization):** Restar la media y dividir por la desviación estándar precalculada del dataset ImageNet (`mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]`). Paso innegociable si se aplica *Transfer Learning*.

### 3.4. Aumento de Datos (Data Augmentation)
Técnicas probabilísticas aplicadas **exclusivamente en la fase de entrenamiento**. Sirven para crear variaciones sintéticas de las imágenes en tiempo real, mejorando la generalización del modelo.

* **Rotaciones Afines Leves:** Girar la imagen aleatoriamente entre `-10°` y `+10°` para simular pacientes que no estaban perfectamente alineados frente a la máquina de rayos X.
* **Modulación de Brillo y Contraste (ColorJitter):** Simular variaciones de exposición generadas por diferentes escáneres hospitalarios.
* **Contraindicación estricta (Volteo Horizontal):** En radiografía torácica, aplicar un *Horizontal Flip* es un error metodológico grave, ya que presenta a la red una condición anatómica anómala e irreal (corazón en el lado derecho o *Situs Inversus*), destruyendo la coherencia espacial del diagnóstico.

### 3.5. Técnicas de Mitigación del Desbalanceo Algorítmico
Ante la imposibilidad de generar radiografías falsas con técnicas clásicas para compensar patologías minoritarias (ej. *Fracturas*), el desbalanceo se corrige matemáticamente durante la optimización.

* **Class Weights (Pesos Positivos):** Multiplicar el error matemático (*Loss*) por un factor penalizador calculado a partir de la prevalencia. Si la red falla en diagnosticar una patología rara, el gradiente de error aplicado es severo.
* **Focal Loss:** Función de pérdida avanzada que reduce dinámicamente la penalización de los ejemplos "fáciles" y concentra el esfuerzo computacional en los ejemplos minoritarios o difíciles de clasificar.

El orden cronológico y estricto que debe seguir en su Jupyter Notebook es este:

1. Limpieza y Mapeo de Etiquetas (Dataframe): Lo primero es convertir los -1.0 y NaN a valores binarios (0.0 o 1.0). El CSV debe quedar inmaculado matemáticamente.

2. Partición Segura (Data Split): Dividir el CSV en Train y Validation agrupando por paciente. [Regla de Oro]: Esto se hace antes de calcular cualquier estadística para evitar contaminar la validación.

3. Cálculo de Pesos (Class Weights): Una vez tiene su CSV de train_df aislado, calcula la prevalencia de las enfermedades solo en ese subconjunto para configurar la función de pérdida.

4. Definición del Pipeline de Imágenes (Transforms): Crear las reglas de transformación (Resize, Normalize, Augmentation).

5. Creación del DataLoader: En PyTorch, las imágenes no se redimensionan y se guardan en el disco duro (ocuparían terabytes). El DataLoader lee la ruta de la imagen, la abre, le aplica las transformaciones del paso 4 y se la da a la red neuronal en lotes (batches) en tiempo real.

### 3.1.1. Orden Cronológico del Data Pipeline en PyTorch

En *Deep Learning*, el preprocesamiento no se ejecuta de golpe sobre todo el dataset (como haríamos en un Excel o con datos tabulares pequeños). Se divide estrictamente en dos fases: transformaciones estáticas sobre el texto y transformaciones dinámicas sobre las imágenes. 

Alterar este orden es la causa principal de *Data Leakage* (Fuga de Datos). El flujo metodológico correcto es el siguiente:

1. **Limpieza y Mapeo de Etiquetas (Metadatos):** Se procesa el DataFrame (`df`) para eliminar nulos y traducir la incertidumbre (`-1.0`) a valores binarios puros (`0.0` o `1.0`). El CSV debe quedar matemáticamente limpio.
2. **Partición Segura (Data Split):** Se divide el DataFrame en *Train*, *Validation* y *Test* utilizando `GroupShuffleSplit` (agrupando por `Patient`). **Regla de oro:** Esto se hace *antes* de calcular cualquier estadística global.
3. **Cálculo de Pesos (Class Weights):** Utilizando **únicamente** el conjunto de entrenamiento aislado (`train_df`), se calcula la prevalencia de las enfermedades para configurar la función de pérdida y penalizar el desbalanceo.
4. **Definición del Pipeline de Imágenes (Transforms):** Se establecen las reglas espaciales y de color usando `torchvision.transforms` (Redimensionado, Normalización ImageNet, Data Augmentation).
5. **Creación del DataLoader:** Las imágenes no se redimensionan y se guardan en el disco duro. El `DataLoader` lee la ruta del CSV, abre la imagen, le aplica las transformaciones del paso 4 en tiempo real y alimenta a la red neuronal por lotes (*batches*).

### 3.1.2. Políticas Clínicas para el Manejo de Incertidumbre (U-Zeroes vs U-Ones)

En el dataset CheXpert, el abanico de valores originales es: `1.0` (Positivo), `0.0` (Negativo), `-1.0` (Incierto) y `NaN` (Celda vacía). 

**¿Qué significa clínicamente el `NaN`?**
Si un radiólogo no menciona una patología en su informe (ej. no habla de fracturas), no es porque tenga dudas, sino porque esa zona anatómica está sana y no es relevante anotarlo. Por tanto, **el `NaN` siempre equivale a `0.0` (Sano)** en cualquier política.

Las dos estrategias principales difieren en cómo tratan la duda médica explícita (`-1.0`):

#### Estrategia U-Zeroes (Recomendada y Pragmática)
Asume que ante la duda del radiólogo, el paciente está sano. Es la estrategia estándar para evitar que el modelo aprenda "ruido" o falsos positivos.
* `1.0` (Enfermo) ➔ Se queda en **`1.0`**
* `0.0` (Sano) ➔ Se queda en **`0.0`**
* `NaN` (No mencionado) ➔ Se convierte en **`0.0`**
* `-1.0` (Incierto) ➔ Se convierte en **`0.0`** #### Estrategia U-Ones (Conservadora)
Asume que ante la duda del radiólogo, es mejor prevenir y tratar al paciente como si estuviera enfermo, maximizando la sensibilidad de la red neuronal.
* `1.0` (Enfermo) ➔ Se queda en **`1.0`**
* `0.0` (Sano) ➔ Se queda en **`0.0`**
* `NaN` (No mencionado) ➔ Se convierte en **`0.0`** *(Nota: Al no haber duda médica, el valor sigue siendo cero).*
* `-1.0` (Incierto) ➔ Se convierte en **`1.0`** ```

Una vez que lo haya integrado en su documento, el siguiente paso natural sería escribir la celda de código de Python que aplica la **Estrategia U-Zeroes** y el **Data Split**. Avíseme cuando esté listo para programar esa parte.